# Hybrid Graph Part 1: Structured Data → Neptune → LlamaIndex Documents

This is Part 1 of the hybrid graph pipeline. It demonstrates how to:

1. Load structured data (CSV, JSON, Excel) into Neptune as typed graph nodes using document-graph
2. Create edges (relationships) between nodes based on foreign keys
3. Convert the Neptune nodes into LlamaIndex `Document` objects with embedded lineage
4. Save the documents for Part 2 (lexical-graph semantic indexing)

## Why This Matters

The hybrid graph combines two complementary capabilities:

- **document-graph** → typed property graph in Neptune (structured queries, traversal, exact match)
- **lexical-graph** → semantic index in OpenSearch (natural language queries, similarity search)

By converting structured nodes to LlamaIndex Documents (with lineage metadata), we enable
semantic search over structured data — then trace results back to the original typed nodes.

## Data Flow

```
CSV/JSON/Excel
    ↓ (document-graph)
Typed nodes in Neptune  ←→  Direct Cypher queries
    ↓ (conversion with lineage)
LlamaIndex Documents
    ↓ (Part 2: lexical-graph)
Semantic index (OpenSearch)  ←→  Natural language queries
    ↓ (lineage correlation)
Back to original typed node in Neptune
```

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph[graphrag]`
- `pip install llama-index-core`
- Neptune cluster endpoint (see 00-Setup)
- Sample data in `data/` directory


In [ ]:
# Setup
import json, os
import pandas as pd
from pathlib import Path
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.document_graph.graph_build import node_to_cypher
from graphrag_toolkit.document_graph import Node

GRAPH_STORE = os.environ.get(
    'GRAPH_STORE',
    'neptune-db://<your-neptune-cluster-endpoint>:8182'
)
TENANT = 'hybrid_demo'

gs = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()
print('Neptune connected')


## Step 1: Load Structured Data into Neptune

Read CSV/JSON/Excel files and write them as typed nodes to Neptune using document-graph's
`node_to_cypher` with tenant scoping. Each file becomes a set of nodes with a specific label
(e.g., `ResearchPaper`, `Author`, `Citation`).

The `flatten_properties` helper ensures Neptune-compatible values (no nested dicts/lists).


In [ ]:
def flatten_properties(props):
    '''Neptune only accepts simple literals.'''
    flat = {}
    for k, v in props.items():
        if isinstance(v, list):
            flat[k] = ', '.join(str(x) for x in v)
        elif isinstance(v, dict):
            flat[k] = json.dumps(v)
        elif v is None:
            continue
        else:
            flat[k] = v
    return flat

data_sources = [
    {'file': 'hybrid_data/data/research_papers.csv', 'type': 'csv', 'node_type': 'ResearchPaper', 'id_field': 'id'},
    {'file': 'hybrid_data/data/authors.json', 'type': 'json', 'node_type': 'Author', 'id_field': 'author_id'},
    {'file': 'hybrid_data/data/citations.csv', 'type': 'csv', 'node_type': 'Citation', 'id_field': 'id'},
]

all_records = {}  # node_type → list of records

for source in data_sources:
    fname = source['file']
    ntype = source['node_type']
    id_field = source['id_field']
    print(f'\nLoading: {fname}')

    if source['type'] == 'csv':
        df = pd.read_csv(fname)
    else:
        raw = json.load(open(fname))
        # Handle nested JSON like {"authors": [...]}
        if isinstance(raw, dict):
            key = list(raw.keys())[0]
            df = pd.DataFrame(raw[key])
        else:
            df = pd.DataFrame(raw)

    records = df.to_dict('records')
    all_records[ntype] = records

    for record in records:
        rid = str(record.get(id_field, hash(str(record))))
        node = Node(id=rid, labels=[ntype], properties=flatten_properties(record))
        cypher, params = node_to_cypher(node, tenant_id=TENANT)
        gs.execute_query(cypher, params)

    print(f'  Wrote {len(records)} {ntype} nodes')

print(f'\nTotal: {sum(len(v) for v in all_records.values())} nodes in Neptune')


## Step 2: Create Relationships (Edges)

Infer and create edges between nodes based on shared identifiers (foreign keys).
For example:
- Papers → `AUTHORED_BY` → Authors (via `author_id`)
- Papers → `CITES` → Papers (via `citation_id`)

This gives us a typed, traversable graph in Neptune.


In [ ]:
# Step 2b: Create edge relationships
from graphrag_toolkit.document_graph.graph_build import edge_to_cypher
from graphrag_toolkit.document_graph import Edge

edges_written = 0

# AUTHORED_BY: link papers to authors (by matching author_id in papers)
papers = all_records.get('ResearchPaper', [])
authors = all_records.get('Author', [])
author_ids = {a.get('author_id') for a in authors}

for paper in papers:
    # Check if paper has an author field referencing an author
    paper_authors = str(paper.get('authors', paper.get('author_id', ''))).split(',')
    for aid in paper_authors:
        aid = aid.strip()
        if aid in author_ids:
            edge = Edge(id=f'e-{paper["id"]}-{aid}', source_id=str(paper['id']), target_id=aid, label='AUTHORED_BY')
            cypher, params = edge_to_cypher(edge, tenant_id=TENANT)
            gs.execute_query(cypher, params)
            edges_written += 1

# CITES: link citations to papers
citations = all_records.get('Citation', [])
paper_ids = {str(p.get('id')) for p in papers}

for cite in citations:
    src = str(cite.get('source_paper', cite.get('citing_paper', '')))
    tgt = str(cite.get('target_paper', cite.get('cited_paper', '')))
    if src and tgt:
        edge = Edge(id=f'c-{cite.get("id", "")}', source_id=src, target_id=tgt, label='CITES')
        cypher, params = edge_to_cypher(edge, tenant_id=TENANT)
        gs.execute_query(cypher, params)
        edges_written += 1

print(f'Wrote {edges_written} edges (AUTHORED_BY + CITES)')


## Step 3: Verify in Neptune

Confirm the graph was built correctly by counting nodes per type.


In [ ]:
result = gs.execute_query('MATCH (n) RETURN labels(n) as lbl, count(n) as cnt')
print('Nodes by type:')
for row in result:
    print(f'  {row}')


## Step 4: Convert to LlamaIndex Documents (with Lineage)

**This is the bridge between document-graph and lexical-graph.**

We convert each typed Neptune node into a LlamaIndex `Document` with:
- **Text**: A structured representation of the node's properties (for semantic embedding)
- **Lineage header**: `[NodeType | node_id | tenant]` embedded in the text

The lineage header survives lexical-graph's chunking process, so after semantic search
you can trace any result back to the original typed node in Neptune.

This is what makes the hybrid graph work: semantic search finds relevant text,
lineage tells you which structured node it came from.


In [ ]:
from llama_index.core.schema import Document

lexical_documents = []

for ntype, records in all_records.items():
    id_field = next(s['id_field'] for s in data_sources if s['node_type'] == ntype)
    for record in records:
        node_id = str(record.get(id_field, 'unknown'))
        source_id = f'{ntype}:{node_id}:{TENANT}'
        # Build text with lineage header
        text = f'[{ntype} | {node_id} | {TENANT}]\n'
        text += '\n'.join(f'{k}: {v}' for k, v in record.items() if v is not None and str(v).strip())
        # Use graphrag-toolkit's source metadata structure
        doc = Document(
            text=text,
            metadata={
                'source': {
                    'sourceId': source_id,
                    'metadata': {
                        'node_type': ntype,
                        'node_id': node_id,
                        'tenant': TENANT,
                    }
                }
            }
        )
        lexical_documents.append(doc)

print(f'Created {len(lexical_documents)} documents')
for ntype in all_records:
    count = sum(1 for d in lexical_documents if d.metadata.get('source', {}).get('metadata', {}).get('node_type') == ntype)
    print(f'  {ntype}: {count}')
print(f'\nSample sourceId: {lexical_documents[0].metadata["source"]["sourceId"]}')
print(f'Sample text:\n{lexical_documents[0].text[:150]}')


## Step 5: Save for Part 2

Serialize the LlamaIndex Documents to disk so Part 2 (`7b-Lexical-Integration-Lexical-Setup`)
can index them into lexical-graph without re-running the data processing.

In production, you'd pass these directly to `LexicalGraphIndex.extract_and_build()`.


In [ ]:
import pickle

output_dir = Path('output/hybrid_foundation')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'lexical_documents.pkl', 'wb') as f:
    pickle.dump(lexical_documents, f)

with open(output_dir / 'summary.json', 'w') as f:
    json.dump({'total_docs': len(lexical_documents), 'node_types': list(all_records.keys()), 'tenant': TENANT}, f, indent=2)

print(f'Saved {len(lexical_documents)} documents to {output_dir}')
print('Ready for 07b — Lexical-Graph Indexing')


# Clean up Neptune connection
gs.__exit__(None, None, None)
